In [48]:
from typing import Annotated # provide additional context without affecting the type itself
from typing import Sequence # automatically handle state update for sequences such as adding messages to a chat history
from typing import TypedDict

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.messages import BaseMessage # foundational class for all message type in LangGraph
from langchain_core.messages import ToolMessage # passes data back to LLM after it call tool(s)
from langchain_core.messages import SystemMessage # message for providing instruction to LLM
from langchain_core.tools import tool

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
import os

from litellm.rust_bridge.ocr.route_host import response

In [49]:
load_dotenv()

API_KEY = os.getenv("API_KEY")
MODEL = os.getenv("MODEL")
BASE_URL = os.getenv("BASE_URL")

In [50]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

@tool
def add(a: int, b: int):
    """ this is an addition function that adds two numbers """
    return a + b

@tool
def subtract(a: int, b: int):
    """ this is an subtraction function that subtracts two numbers """
    return a - b

@tool
def multiply(a: int, b: int):
    """ this is an multiplication function that multiply two numbers """
    return a * b

tools = [add, subtract, multiply]

In [51]:
llm = ChatOpenAI(model=MODEL, api_key=API_KEY, base_url=BASE_URL)
llm = llm.bind_tools(tools)


In [52]:
def model_call(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content="You are my AI assistant, answer my query to the best of your ability")
    response = llm.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

In [53]:
def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

In [54]:
OUR_AGENT_NODE = "our_agent"
TOOLS_NODE = "tools"

graph = StateGraph(AgentState)
graph.add_node(OUR_AGENT_NODE, model_call)

tool_node = ToolNode(tools=tools)
graph.add_node(TOOLS_NODE, tool_node)

graph.set_entry_point(OUR_AGENT_NODE)
graph.add_conditional_edges(
    OUR_AGENT_NODE,
    should_continue,
    {
        "continue": TOOLS_NODE,
        "end": END
    }
)

graph.add_edge(TOOLS_NODE, OUR_AGENT_NODE)

app = graph.compile()

In [56]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]

        if hasattr(message, "tool_calls") and message.tool_calls:
            print(f"\n[Node: Agent] -> Requesting Tool Execution:")
            for tc in message.tool_calls:
                print(f"   🔧 Tool: {tc['name']} | Args: {tc['args']}")
        elif hasattr(message, "type") and message.type == "tool":
            print(f"\n[Node: Tools] -> Tool Executed Successfully:")
            print(f"   📥 Result: {message.content}")
        else:
            print(f"\n[Node: Agent] -> Text Response:")
            message.pretty_print()

inputs = {"messages": [("user", "add 31 + 53, add 4 + 3 and multiply the result by 2, subtract 99 - 9")]}
print_stream(app.stream(inputs, stream_mode="values"))


[Node: Agent] -> Text Response:
================================ Human Message =================================

add 31 + 53, add 4 + 3 and multiply the result by 2, subtract 99 - 9

[Node: Agent] -> Requesting Tool Execution:
   🔧 Tool: add | Args: {'a': 31, 'b': 53}

[Node: Tools] -> Tool Executed Successfully:
   📥 Result: 84

[Node: Agent] -> Requesting Tool Execution:
   🔧 Tool: add | Args: {'b': 3, 'a': 4}

[Node: Tools] -> Tool Executed Successfully:
   📥 Result: 7

[Node: Agent] -> Requesting Tool Execution:
   🔧 Tool: multiply | Args: {'b': 2, 'a': 7}

[Node: Tools] -> Tool Executed Successfully:
   📥 Result: 14

[Node: Agent] -> Requesting Tool Execution:
   🔧 Tool: subtract | Args: {'a': 99, 'b': 9}

[Node: Tools] -> Tool Executed Successfully:
   📥 Result: 90

[Node: Agent] -> Text Response:
================================== Ai Message ==================================

Here are the results of each step you asked for:

1. **31 + 53 = 84**  
2. **4 + 3 = 7**  
3. **7 × 2